# Nested model comparison: KS tests and AIC

Compares nested WITCH fits of the same data, ie: MOOJ1142 with r_s fixed, r_s free, and the full gNFW shape free.

**Method**
* Each run is loaded from its output directory (`config.yaml`, `results_mcmc.dill`, `samples_mcmc.npz`).
* Every pair of runs is checked to be nested, `witch.model_comparison.NestingError` is raised if not:
  same data and likelihood config, same structures and ties, the smaller run's free parameters are a subset of
  the larger run's, parameters fixed in both have the same value, and every parameter only the larger run
  frees is fixed (in the smaller run) inside the larger run's prior.
* **KS:** two-sample KS test on the posterior of each shared free parameter. The p-value uses the effective
  sample size of the chains since MCMC samples are correlated, and a `ChainQualityError` is raised if any
  parameter has fewer than `MIN_ESS` effective samples.
* **AIC:** every model is evaluated against **one fixed noise model, the one estimated from the raw data before
  any model subtraction**, so the chi-squared values come from the same likelihood. Each model is evaluated at the
  median of its posterior. AIC = chisq + 2k with k the number of free parameters.

The fits themselves re-estimate the noise from their own residuals after each LM round, so the chi-squared
saved with each run is not comparable between runs, which is why the data is reloaded here.

**Where to run:** the KS part only reads the chains. The AIC part loads all the TODs in a single process
(roughly 0.3 GB per TOD plus ~4 GB for the compiled model), so run it on a compute node, not a login node.
`DEBUG = True` loads only a few TODs for a quick end to end check; its chi-squared values are not meaningful.


In [ ]:
# Runs to compare, label -> output directory. Order doesn't matter, they are sorted by number of free parameters.
REDUCTIONS = "/scratch/jorlo/MUSTANG/Reductions/MOOJ1142"
RUNS = {
    "fixed_rs": f"{REDUCTIONS}/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs/r-beta-amp-P0-beta-dx_g-dy_g-sigma-amp_g",
    "free_rs": f"{REDUCTIONS}/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs/r-beta-amp-P0-r_s-beta-dx_g-dy_g-sigma-amp_g",
    "all_free": f"{REDUCTIONS}/mustang2/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs_all_free/r-beta-amp-P0-r_s-gamma-alpha-beta-dx_g-dy_g-sigma-amp_g",
}
WITCH_DATROOT = "/scratch/jorlo/MUSTANG/M2-TODs"
MIN_ESS = 50  # Minimum effective samples per parameter for the KS test
RUN_AIC = True  # Set False to only do the KS part (no data loading)
DEBUG = False  # Only load DEBUG_NTODS TODs, chisq is then NOT comparable to a full fit
DEBUG_NTODS = 4
FIGDIR = "figures"


In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ["WITCH_DATROOT"] = WITCH_DATROOT
os.makedirs(FIGDIR, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from witch import model_comparison as wmc

pd.set_option("display.float_format", lambda x: f"{x:.6g}")


In [ ]:
runs = [wmc.load_run(outdir, label) for label, outdir in RUNS.items()]
# Raises NestingError if any pair isn't nested
runs = wmc.order_nested(runs)
display(pd.DataFrame([{"run": r.label, "k": r.k, "chains": r.chains.shape, "outdir": r.outdir} for r in runs]))
for small, big in zip(runs[:-1], runs[1:]):
    print(f"{small.label} -> {big.label} frees: {wmc.check_nested(small, big)}")


## KS tests on the shared free parameters

`D` is the KS statistic (max distance between the CDFs), `p` the ESS corrected p-value. A small `p` means the
marginal posterior of that parameter moved when the extra parameters were freed; this says nothing about which model
fits better (that is what the AIC below is for).


In [ ]:
import itertools

ks_tables = {}
for small, big in itertools.combinations(runs, 2):
    ks_tables[(small.label, big.label)] = df = pd.DataFrame(wmc.ks_compare(small, big, MIN_ESS))
    print(f"{small.label} vs {big.label}")
    display(df)


In [ ]:
shared = runs[0].free_names  # Free in every run since the runs are nested
ncol = 3
fig, axes = plt.subplots(int(np.ceil(len(shared) / ncol)), ncol, figsize=(4 * ncol, 3 * np.ceil(len(shared) / ncol)))
for ax, name in zip(axes.flat, shared):
    for run in runs:
        ax.hist(run.chains[..., run.free_names.index(name)].ravel(), bins=60, density=True, histtype="step", label=run.label)
    ax.set_xlabel(name.split(".", 1)[1])
for ax in list(axes.flat)[len(shared):]:
    ax.set_visible(False)
axes.flat[0].legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "nested_posteriors.png"), dpi=150)


## AIC with the initial noise model

The data is loaded once with the config of the largest model and the noise is estimated from the raw TODs.
Every run is then evaluated on that same data and noise at its posterior medians.


In [ ]:
if RUN_AIC:
    # Data settings from the largest run's config, model from its saved metamodel
    metamodel = wmc.load_initial_noise_data(runs[-1], ntods=DEBUG_NTODS if DEBUG else None)
    chisq = wmc.ChisqEvaluator(metamodel)
    chisqs = [chisq(run) for run in runs]
    for warning in wmc.chisq_order_warnings(runs, chisqs):
        print("WARNING, larger model fits worse at the medians:", warning)
    aic = pd.DataFrame(wmc.aic_table(runs, chisqs))
    if DEBUG:
        print(f"DEBUG: only {DEBUG_NTODS} TODs loaded, these numbers are not meaningful")
    display(aic)


In [ ]:
if RUN_AIC:
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(aic["run"], aic["dAIC"])
    ax.axhline(10, color="k", ls="--", lw=1, label=r"$\Delta$AIC = 10")
    ax.set_ylabel(r"$\Delta$AIC")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, "nested_daic.png"), dpi=150)


## Reading the results
* ΔAIC is relative to the best (lowest AIC) run. ΔAIC > 10 means essentially no support for that model,
  2–10 is positive to strong evidence against it, < 2 means the models are indistinguishable.
* The AIC is formally defined at the maximum likelihood; the medians are used here as requested. If a larger model
  ends up with a higher chi-squared than a model nested in it, a warning is printed above, this usually means
  a skewed posterior or a poorly converged chain.

To make an HTML report on a compute node:
`jupyter nbconvert --to html --execute notebooks/nested_model_comparison.ipynb`
